In [20]:
# Some utitity libraries to test optuna
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Optuna is the library used for hyperparameter estimation based on many samplers.
# Mostly we use a sampler called TPE.
# Now this notebook is a complete walkthrough on why do we need optuna ? How did it come to existence ?
# Why other sampling methods failed ? 
# This notebook is not just about simply implement optuna -- It is about comprehending it's need.

In [ ]:
# Way before we had no such things as optimizer . All we had  was intutive guessing and manual looping over different hyperparameters.
# This was manual tuning .The code block would look like: 

In [11]:
# A simple iris dataset 
X,y = load_iris(return_X_y=True)

# Divide the data into three set of data
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
X_test,X_val,y_test,y_val = train_test_split(X_test,y_test,test_size=0.5,random_state=42)

In [ ]:
depths = [2,3,4,5,6,7,8]

best_score = -1
best_depth = None

for depth in depths:

    model = RandomForestClassifier(max_depth=depth)

    model.fit(X_train, y_train)

    score = model.score(X_val, y_val)

    print(f"Depth : {depth} | Score : {score}")

    if score > best_score:
        best_score = score
        best_depth = depth

print(best_depth)

# NOTE : This is codeblock is not to be evaluated but instead it showcases the manual hardship we had to do get the perfect the parameter.
# The dataset is too simple and model can easily learn all the patterns.

In [ ]:
# This was just a few depths and also one parameter .
# Now imagine big data , 100 of parameters and wait we have not talked about the combinations of each parameter yet.
# Multiple nested loops , memory hell and infinite-like time --- This is what we had to deal with for many years.

# We could have guessed parameters but there is a limit to the intutive guessing.
# Our function is basically: 

# From its perspective, it is solving        :   f(hyperparameters) → validation score
# The entire optimisation problem is simply  :   Find the hyperparameters that maximise (or minimise) this unknown function.

In [24]:
# Same set up as previous but this time we use kfold splitting to avoid data leaks , and test the values at the end.
# We play with training set using kfold to train and find the best parameters

X, y = load_iris(return_X_y=True)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Parameter grid hash to search from
param_grid = {
    'n_estimators'  : [50,75,100,125],  
    'max_depth'     : [3,5,7,None] 
    }
# Here we have 4 combinations for each param then there will be 4*4 combinations to search from.

# Store the best score and best parameters globally
best_score = 0
best_params = {}

# The number of loops is n+1 ; where n is no of parameters and an extra loop for k fold split
# Basically we will average the scores at the end for each split -- to avoid data leaks but also a generalized model which does not overfits.

for n_est in param_grid['n_estimators']:
    for depth in param_grid['max_depth']:
        
        # Set up K-Fold to permute the Train/Val data
        kf = KFold(n_splits=4, shuffle=True, random_state=42)
        fold_scores = []
        
        # Manually split and place according to training and validation indexes from train set till there is no split left.
        for train_idx, val_idx in kf.split(X_train_val): # The number of types we will split is the same times as n_splits passed for KFold class.
            X_train, X_val = X_train_val[train_idx], X_train_val[val_idx]
            y_train, y_val = y_train_val[train_idx], y_train_val[val_idx]
            
            # Initialize Random Forest with current combination
            model = RandomForestClassifier(
                n_estimators=n_est, 
                max_depth=depth, 
                random_state=42  # Kept constant for fair comparison
            )
            
            # Train and validate
            model.fit(X_train, y_train)
            score = model.score(X_val, y_val)
            fold_scores.append(score)
            
        # Average validation score across all 4 folds (no of loops)
        avg_val_score = np.mean(fold_scores)
        print(f"Avg Val Accuracy: {avg_val_score:.4f} | Params: n_estimators={n_est}, max_depth={depth}")
        
        # Save the best parameters by evaluating from the prev avg scores saved.
        if avg_val_score > best_score:
            best_score = avg_val_score
            best_params = {'n_estimators': n_est, 'max_depth': depth}

print("\n--- Grid Search Complete ---")
print(f"Best Hyperparameters found: {best_params} with Val Accuracy: {best_score:.4f}")

# Train final model on ALL Train/Val data using best params
final_model = RandomForestClassifier(**best_params, random_state=42)
final_model.fit(X_train_val, y_train_val)

# Test exactly ONCE on the locked-away Test Stack
test_accuracy = final_model.score(X_test, y_test)
print(f"Final Accuracy on Locked Test Stack: {test_accuracy:.4f}")


Avg Val Accuracy: 0.9667 | Params: n_estimators=50, max_depth=3
Avg Val Accuracy: 0.9667 | Params: n_estimators=50, max_depth=5
Avg Val Accuracy: 0.9667 | Params: n_estimators=50, max_depth=7
Avg Val Accuracy: 0.9667 | Params: n_estimators=50, max_depth=None
Avg Val Accuracy: 0.9750 | Params: n_estimators=75, max_depth=3
Avg Val Accuracy: 0.9583 | Params: n_estimators=75, max_depth=5
Avg Val Accuracy: 0.9500 | Params: n_estimators=75, max_depth=7
Avg Val Accuracy: 0.9500 | Params: n_estimators=75, max_depth=None
Avg Val Accuracy: 0.9667 | Params: n_estimators=100, max_depth=3
Avg Val Accuracy: 0.9667 | Params: n_estimators=100, max_depth=5
Avg Val Accuracy: 0.9583 | Params: n_estimators=100, max_depth=7
Avg Val Accuracy: 0.9583 | Params: n_estimators=100, max_depth=None
Avg Val Accuracy: 0.9750 | Params: n_estimators=125, max_depth=3
Avg Val Accuracy: 0.9583 | Params: n_estimators=125, max_depth=5
Avg Val Accuracy: 0.9583 | Params: n_estimators=125, max_depth=7
Avg Val Accuracy: 0.9583

In [26]:
# This is a simple hardcoded working of how  gridsearch csv works behind the scenes.
# Look closely how we working a grid like pattern : 
#   First we go on with first parameters and keep on looping till all the nested loop combinations are finished , exiting and going on to another outer loop and so on.

# The similarities and difference between ManualTuning and GridSeachCV
------------------------------
## The Similarity (The Logic)
Both methods do the exact same work behind the scenes:

   1. They take a dictionary of hyperparameters.
   2. They map out every single possible combination (the "grid").
   3. They run cross-validation loops for every combination.
   4. They pick the combination with the highest average validation score. [3, 4, 5, 6, 7] 

------------------------------
## The Differences (The Execution)

| Feature | Manual Tuning (Your Loop Code) | Scikit-Learn's GridSearchCV |
|---|---|---|
| Code Length | Requires 30+ lines of nested loops and index tracking. | Requires only 3 to 4 lines of clean code. |
| Error Proneness | Easy to accidentally leak data or mess up index slicing. | Robust and thoroughly tested to prevent bugs. |
| Speed (Parallelization) | Runs on a single CPU core sequentially (slow). | Can run on all CPU cores at once using n_jobs=-1 (very fast). |
| Refitting | You must manually write code to retrain the model on the full data at the end. | Automatically retrains the best model for you (refit=True). |

------------------------------


In [27]:
# NOTE :  We did not import the GridSearchCV of the scikit-learn , which is much more robust.